# NB03 — LSTM Deep Momentum Network

**Pergam adaptation**: long-only sigmoid [0, 1], 25 bps transaction costs, ~600 European equities.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs" / "default.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.nb03_dmn import (
    seed_everything, load_nb03_inputs, prepare_model_data,
    input_summary, feature_summary, architecture_summary, loss_summary,
    variant_table, make_folds, run_all_variants, enrich_diagnostics,
    variant_summary, plot_sharpe_by_fold, plot_turnover_by_fold,
    plot_position_distribution, plot_loss_curves, plot_example_ticker,
    save_nb03_outputs, load_existing_outputs,
)

seed_everything(42)


## 1. Inputs


In [ ]:
data       = load_nb03_inputs(PROJECT_ROOT)
model_data = prepare_model_data(data["panel"], data["cpd_scores"])

display(input_summary(data, model_data))
display(feature_summary(model_data))


## 2. Architecture


In [ ]:
display(architecture_summary())
display(variant_table(model_data))


## 3. Loss — 25 bps (Pergam)


In [ ]:
display(loss_summary())


## 4. Walk-forward Folds


In [ ]:
folds = make_folds(model_data)
display(folds[["fold", "train_years", "validation_year", "test_year", "test_start", "test_end"]])


## 5. Training


In [ ]:
dmn_positions, dmn_metrics, dmn_diagnostics = run_all_variants(
    model_data, folds,
    variants=["baseline", "cpd_cusum", "cpd_gp", "cpd_bocpd"],
    checkpoint=True,
    root=PROJECT_ROOT,
)
dmn_diagnostics = enrich_diagnostics(dmn_positions, dmn_diagnostics)
display(variant_summary(dmn_diagnostics))


## 6. Diagnostics


In [ ]:
plot_sharpe_by_fold(dmn_diagnostics).show()


In [ ]:
plot_turnover_by_fold(dmn_diagnostics).show()


In [ ]:
plot_position_distribution(dmn_positions).show()


## 7. Loss Curves


In [ ]:
for variant in ["baseline", "cpd_cusum", "cpd_gp", "cpd_bocpd"]:
    if variant in dmn_metrics["variant"].unique():
        plot_loss_curves(dmn_metrics, variant).show()


## 8. Example Ticker


In [ ]:
plot_example_ticker(dmn_positions, model_data, data["known_events"]).show()


## 9. Save


In [ ]:
display(save_nb03_outputs(PROJECT_ROOT, dmn_positions, dmn_metrics, dmn_diagnostics))
display(variant_summary(dmn_diagnostics))
